In [1]:
import numpy as np
import pandas as pd
import duckdb

In [2]:
df = pd.read_csv('./data/1mil2-isrc.csv')
df.shape

(1204025, 25)

In [3]:
df.rename(columns={"name": "song"}, inplace=True)

In [4]:
df.isna().sum()

id                       0
song                     3
album                   11
album_id                 0
artists                  0
artist_ids               0
track_number             0
disc_number              0
explicit                 0
danceability             0
energy                   0
key                      0
loudness                 0
mode                     0
speechiness              0
acousticness             0
instrumentalness         0
liveness                 0
valence                  0
tempo                    0
duration_ms              0
time_signature           0
year                     0
release_date             0
isrc                457659
dtype: int64

In [2]:
df_hot = pd.read_csv('./data/hot-100-2000.csv')
df_hot.shape

(114100, 8)

In [3]:
df_hot.rename(columns={"artist": "artists"}, inplace=True)

In [7]:
df_hot.isna().sum()

date                  0
rank                  0
song                  0
artists               0
last-week         11005
peak-rank             0
weeks-on-board        0
isrc               1122
dtype: int64

In [6]:
df_hot['isrc'].isin(df['isrc']).sum()

np.int64(29882)

In [6]:
df.to_csv("./data/1mil2-isrc.csv", index = False)

In [4]:
df_hot.to_csv("./data/hot-100-2000.csv", index=False)

In [9]:
lyr = pd.read_csv("./data/song_lyrics.csv", usecols=["title", "artist", "year", "features", "id"])
lyr.shape

(5134856, 5)

In [10]:
lyr.rename(columns={"title": "song"}, inplace=True)

## Normalization

In [11]:
from normalize import norm_billboard, norm_spotify, add_norm_artists_lyrics
df["norm_artists"] = df["artists"].map(norm_spotify)
df_hot["norm_artists"]=df_hot["artists"].map(norm_billboard)
add_norm_artists_lyrics(lyr)

,song,artist,year,features,id,norm_artists
0,Killa Cam,Cam'ron,2004,"{""Cam\\'ron"",""Opera Steve""}",1,"frozenset({operastevecamron, camron, operastev..."
1,Can I Live,JAY-Z,1996,{},3,frozenset({jayz})
2,Forgive Me Father,Fabolous,2003,{},4,frozenset({fabolous})
3,Down and Out,Cam'ron,2004,"{""Cam\\'ron"",""Kanye West"",""Syleena Johnson""}",5,"frozenset({syleenajohnsonkanyewestcamron, camr..."
4,Fly In,Lil Wayne,2005,{},6,frozenset({lilwayne})
...,...,...,...,...,...,...
5134851,Ocean,Effemar,2022,{},7882842,frozenset({effemar})
5134852,64 Bars,Rapido,2022,{},7882843,frozenset({rapido})
5134853,Raise Our Hands,"Culture Code, Pag & Mylo",2016,"{Elex,""Culture Code / Pag & Mylo""}",7882845,"frozenset({culturecodeelexpagmylo, mylo, myloc..."
5134854,CEO,Antropolita,2022,{},7882846,frozenset({antropolita})


## Match count for 114k and 1mil2

In [15]:
def _title_key(series):
    return series.astype(str).str.lower().str.strip()


def _explode(frame, title_col, positions):
    """One row per (title, artist variant); `pos` = row position in `frame`."""
    out = pd.DataFrame({
        "title_key": _title_key(frame[title_col]).to_numpy()[positions],
        "variant": frame["norm_artists"].to_numpy()[positions],
        "pos": positions,
    })
    out["variant"] = out["variant"].map(lambda s: list(s) if s else [])
    return out.explode("variant").dropna(subset=["variant"])


def count_matches(df, df_hot, song_col="song", artist_col="artists"):
    """Count Billboard (df_hot) rows/songs that have a match in df (1mil2).
    Match = same lowercased title AND norm_artists sets share a variant."""
    hot = _explode(df_hot, song_col, np.arange(len(df_hot)))

    # only keep 1mil2 rows whose title exists in Billboard (keeps explode small)
    df_keys = _title_key(df[song_col])
    cand = np.flatnonzero(df_keys.isin(set(hot["title_key"])).to_numpy())
    spot = _explode(df, song_col, cand)

    pairs = (
        hot.merge(spot, on=["title_key", "variant"], suffixes=("_hot", "_df"))
        [["pos_hot", "pos_df"]]
        .drop_duplicates()
        .rename(columns={"pos_hot": "df_hot_pos", "pos_df": "df_pos"})
    )

    matched = np.zeros(len(df_hot), dtype=bool)
    matched[pairs["df_hot_pos"].unique()] = True

    n_rows, n_rows_matched = len(df_hot), int(matched.sum())
    songs = pd.DataFrame({
        "title_key": _title_key(df_hot[song_col]).to_numpy(),
        "artist": df_hot[artist_col].to_numpy(),
        "matched": matched,
    })
    per_song = songs.groupby(["title_key", "artist"])["matched"].max()
    n_songs, n_songs_matched = len(per_song), int(per_song.sum())

    print(f"Billboard rows : {n_rows_matched:,} / {n_rows:,} matched ({n_rows_matched / n_rows:.1%})")
    print(f"Unique songs   : {n_songs_matched:,} / {n_songs:,} matched ({n_songs_matched / n_songs:.1%})")
    versions = pairs.groupby("df_hot_pos")["df_pos"].nunique()
    if len(versions):
        print(f"1mil2 rows per matched Billboard row: median {versions.median():.0f}, max {versions.max()}")

    return pairs, matched

In [18]:
pairs, matched = count_matches(df, df_hot)

Billboard rows : 28,122 / 114,100 matched (24.6%)
Unique songs   : 1,898 / 9,197 matched (20.6%)
1mil2 rows per matched Billboard row: median 1, max 13


## Match count for 1mil2 and lyr

In [ ]:
def _title_key(series):
    return series.astype(str).str.lower().str.strip()
 
 
def _explode(frame, title_keys, positions):
    """One row per (title, artist variant); `pos` = row position in `frame`."""
    out = pd.DataFrame({
        "title_key": title_keys[positions],
        "variant": frame["norm_artists"].to_numpy()[positions],
        "pos": positions,
    })
    out["variant"] = out["variant"].map(lambda s: list(s) if s else [])
    return out.explode("variant").dropna(subset=["variant"])
 
 
def count_matches_lyr(left, right, song_col="song", artist_col="artists",
                  year_col="year", year_tol=1):
    """
    How many rows of `left` have a match in `right`.
    Match = same lowercased title AND norm_artists share a variant
            AND |year_left - year_right| <= year_tol   (skipped if year_col=None)
 
    Returns (pairs, matched):
      pairs   DataFrame: left_pos, right_pos, year_diff (positions, use .iloc)
              (all title+artist matches, year_diff lets you choose the window)
      matched boolean array aligned with `left` (matches within year_tol)
    """
    lkeys = _title_key(left[song_col]).to_numpy()
    rkeys = _title_key(right[song_col]).to_numpy()
 
    # only explode rows whose title exists on both sides
    common = pd.Index(lkeys).unique().intersection(pd.Index(rkeys).unique())
    lpos = np.flatnonzero(pd.Series(lkeys).isin(common).to_numpy())
    rpos = np.flatnonzero(pd.Series(rkeys).isin(common).to_numpy())
 
    L = _explode(left, lkeys, lpos)
    R = _explode(right, rkeys, rpos)
 
    pairs = (
        L.merge(R, on=["title_key", "variant"], suffixes=("_l", "_r"))
        [["pos_l", "pos_r"]]
        .drop_duplicates()
        .rename(columns={"pos_l": "left_pos", "pos_r": "right_pos"})
        .reset_index(drop=True)
    )
 
    n = len(left)
    if year_col:
        ly = pd.to_numeric(left[year_col], errors="coerce").to_numpy(dtype=float)
        ry = pd.to_numeric(right[year_col], errors="coerce").to_numpy(dtype=float)
        pairs["year_diff"] = np.abs(ly[pairs["left_pos"]] - ry[pairs["right_pos"]])
        ok = pairs[pairs["year_diff"] <= year_tol]
    else:
        pairs["year_diff"] = np.nan
        ok = pairs
 
    def _rows(p):
        return p["left_pos"].nunique()
 
    matched = np.zeros(n, dtype=bool)
    matched[ok["left_pos"].unique()] = True
 
    # unique songs (title + raw artist string)
    songs = pd.DataFrame({"t": lkeys, "a": left[artist_col].to_numpy(), "m": matched})
    per_song = songs.groupby(["t", "a"])["m"].max()
 
    print(f"left rows: {n:,}")
    print(f"  title + artist only      : {_rows(pairs):,} ({_rows(pairs) / n:.1%})")
    if year_col:
        exact = pairs[pairs["year_diff"] == 0]
        print(f"  + same year              : {_rows(exact):,} ({_rows(exact) / n:.1%})")
        print(f"  + year within +/-{year_tol}       : {_rows(ok):,} ({_rows(ok) / n:.1%})")
    print(f"  unique songs matched     : {int(per_song.sum()):,} / {len(per_song):,} "
          f"({per_song.mean():.1%})")
    per_row = ok.groupby("left_pos")["right_pos"].nunique()
    if len(per_row):
        print(f"  right rows per matched left row: median {per_row.median():.0f}, max {per_row.max()}")
 
    return pairs, matched

In [ ]:
pairs, matched = count_matches_lyr(df, lyr)